# Limpeza de Dados: Goodreads Dataset
Esta etapa trata exclusivamente da **higienização estrutural** do dataset bruto do Goodreads, preparando-o para consumo seguro nas etapas de Engenharia de Features e Análise. Nenhuma inteligência analítica é criada aqui — o foco é garantir que cada linha que avança no pipeline represente um registro real, confiável e bem tipado.

- **Escopo:** Remoção de ruídos, tratamento de nulos e correção de tipos primitivos (`date`, `float`)
- **Produto desta etapa:** `goodreads_books_cleaned.csv` — a fonte de verdade para os notebooks subsequentes

In [1]:
# Configuração do Jupyter (Autoreload)
%load_ext autoreload
%autoreload 2

# Configuração de Caminho (Path Setup)
import sys
import os

# Adiciona a pasta raiz do projeto (..) ao sistema para liberar os imports locais
sys.path.append(os.path.abspath(os.path.join('..', '..')))


# Importação de Bibliotecas e Módulos
import pandas as pd

# Nossos módulos customizados da pasta src/
import src.io.data_loader as dl
import src.io.data_save as ds
import src.view.tables as tb
import src.utils.cleaners as cle

---

## Carregando Dados

In [2]:
caminho = '../../data/raw/books/Goodreads_books_with_genres.csv'
df_books = dl.load_data(caminho, tipo_arquivo='csv')

Dados CSV carregados! Formato: (11127, 13)


---
### Critério de Descarte: Irrelevância para o Escopo Analítico

A coluna `isbn13` é um identificador de catálogo editorial sem valor analítico para os objetivos deste projeto — que visam padrões de recepção, popularidade e gênero literário. Sua presença apenas aumentaria a cardinalidade de identificadores únicos sem contribuir com nenhuma dimensão de análise.

In [3]:
# Dropando a coluna 'isbn13' que não é necessária para a análise
df_clean = df_books.drop(columns=['isbn13'])

---
## Padronização de colunas 
As colunas `Title` e `Book_id` são de estrema importância para este dataset, entretanto seus nomes se destacam das demais do schema, gerando anomalias visuais, uniformizar as colunas para `title` e `book_id`respectivamenete, poupará tempo de codagem, e deixará as visualisaçãoes e análises mais simples e claras. 

In [4]:
# Renomeando Coluna 'Title'
df_clean.rename(columns={'Title':'title'}, inplace=True)

In [5]:
# Renomeando Coluna 'Book Id'
df_clean.rename(columns={'Book Id':'book_id'}, inplace=True)

---
## Correção de Tipo: Integridade Temporal do Dataset

A coluna `publication_date` precisa ser tratada como um tipo temporal nativo para viabilizar qualquer análise de séries históricas, como evolução de popularidade por década ou sazonalidade de publicações. Armazenada como string, ela seria invisível para qualquer operação de filtragem ou agrupamento por período.

In [6]:
# Converte a coluna de texto para o tipo Data (datetime)
df_clean['publication_date'] = pd.to_datetime(df_clean['publication_date'], errors='coerce')

---
## Aplicando Limpeza de Missing Data 

### Filtro de Engajamento Real: Eliminação de Registros Fantasma

Livros com `average_rating = 0` ou `ratings_count = 0` não representam obras sem leitores — representam **registros incompletos** que nunca receberam interação verificável na plataforma. Mantê-los distorceria métricas de distribuição de notas e correlações de popularidade.

- **Regra aplicada:** Somente registros com engajamento comprovado (ao menos 1 avaliação) avançam no pipeline
- **Impacto:** Elimina o viés de registros cadastrados mas nunca consumidos pelo público

In [7]:
# Checando as colunas do DataFrame para confirmar a remoção da coluna 'isbn13'
df_clean.columns

Index(['book_id', 'title', 'Author', 'average_rating', 'isbn', 'language_code',
       'num_pages', 'ratings_count', 'text_reviews_count', 'publication_date',
       'publisher', 'genres'],
      dtype='object')

In [8]:
# Remover linhas com valores nulos na coluna de gêneros
df_clean = df_clean.dropna(subset=['genres'])

# Aplicação do filtro de engajamento real utilizando a sintaxe otimizada .query()
df_clean = df_clean.query("average_rating > 0.0 and ratings_count > 0")

# Removendo as linhas onde a coluna 'publication_date' tem valores nulos (NaT)
df_clean = df_clean.dropna(subset=['publication_date'])

# Remove os livros com páginas zeradas (audiobooks/coleções)
df_clean = df_clean[df_clean['num_pages'] > 0]

# Remove registros onde o autor foi classificado explicitamente como 'NOT A BOOK'
df_clean = df_clean[df_clean['Author'] != 'NOT A BOOK']


print(f"Limpeza concluída! Tamanho do dataset original: {len(df_books)}")
print(f"Tamanho do dataset limpo: {len(df_clean)}")
print(f"Total de linhas removidas: {len(df_books) - len(df_clean)}")


Limpeza concluída! Tamanho do dataset original: 11127
Tamanho do dataset limpo: 10897
Total de linhas removidas: 230


---
## Deserialização de Colunas Estruturadas: De JSON Serializado para Listas Nativas

A Coluna `genres` sai do CSV como strings que representa uma estrutura JSON — um formato que preserva a estrutura original, mas torna os dados **opacos para qualquer operação analítica**. Sem a extração dos valores internos, é impossível filtrar por gênero, agrupar por país de produção ou identificar pertencimento a franquias.

- **Decisão arquitetural:** A extração é centralizada em funções reutilizáveis para garantir tratamento uniforme e seguro de nulos da coluna.


In [9]:
df_clean['genres'] = df_clean['genres'].apply(
    cle.extrair_lista_dicts, 
    valor_padrao="Gênero Não Identificado"
)

In [10]:
display(tb.estilizar_tabela(
    df=df_books,
    colunas_selecionadas=['title', 'genres'],
    qtd_linhas=20,
    caption="Visualizando gerênos em formato de lista"
))

,genres
0,"Fantasy;Young Adult;Fiction;Fantasy,Magic;Childrens;Adventure;Audiobook;Childrens,Middle Grade;Classics;Science Fiction Fantasy"
1,"Fantasy;Young Adult;Fiction;Fantasy,Magic;Childrens;Adventure;Audiobook;Childrens,Middle Grade;Classics;Science Fiction Fantasy"
2,"Fantasy;Fiction;Young Adult;Fantasy,Magic;Childrens;Childrens,Middle Grade;Audiobook;Adventure;Classics;Science Fiction Fantasy"
3,"Fantasy;Fiction;Young Adult;Fantasy,Magic;Childrens;Childrens,Middle Grade;Adventure;Audiobook;Classics;Science Fiction Fantasy"
4,"Fantasy;Young Adult;Fiction;Fantasy,Magic;Adventure;Fantasy,Supernatural;Mystery;Childrens;Fantasy,Paranormal;Childrens,Middle Grade"
5,Fiction
6,"Fantasy;Fiction;Young Adult;Fantasy,Magic;Childrens;Classics;Adventure;Science Fiction Fantasy;Novels;Paranormal,Wizards"
7,"Science Fiction;Fiction;Humor;Fantasy;Classics;Humor,Comedy;Science Fiction Fantasy;Adventure;Novels;European Literature,British Literature"
8,"Science Fiction;Fiction;Humor;Fantasy;Classics;Humor,Comedy;Science Fiction Fantasy;Adventure;Novels;European Literature,British Literature"
9,"Science Fiction;Fiction;Humor;Classics;Fantasy;Humor,Comedy;Science Fiction Fantasy;Audiobook;Adventure;Novels"


In [11]:
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
Index: 10897 entries, 0 to 11126
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   book_id             10897 non-null  int64         
 1   title               10897 non-null  object        
 2   Author              10897 non-null  object        
 3   average_rating      10897 non-null  float64       
 4   isbn                10897 non-null  object        
 5   language_code       10897 non-null  object        
 6   num_pages           10897 non-null  int64         
 7   ratings_count       10897 non-null  int64         
 8   text_reviews_count  10897 non-null  int64         
 9   publication_date    10897 non-null  datetime64[ns]
 10  publisher           10897 non-null  object        
 11  genres              10897 non-null  object        
dtypes: datetime64[ns](1), float64(1), int64(4), object(6)
memory usage: 1.1+ MB


---

## Salvando Novo Dataset Totatalmente Limpo

In [12]:
# Salvando o dataset limpo para a próxima etapa de análise
ds.save_dataset(
    df=df_clean,
    pasta='../../data/interim/books', 
    nome_arquivo='goodreads_books_cleaned',
    tipo_arquivo='parquet'
)

Sucesso! Ficheiro guardado em '..\..\data\interim\books\goodreads_books_cleaned.parquet'


---
## Conclusão da Limpeza de Dados

O dataset bruto do Goodreads foi higienizado e está pronto para consumo analítico. As intervenções realizadas nesta etapa garantiram três propriedades fundamentais de qualidade:

- **Relevância:** Colunas sem valor para o escopo do projeto foram descartadas, reduzindo dimensionalidade desnecessária
- **Veracidade:** Registros sem engajamento verificável foram removidos para assegurar que métricas de recepção reflitam opiniões reais de leitores
- **Tipagem correta:** Campos temporais e numéricos foram convertidos para seus tipos nativos, habilitando operações analíticas precisas nas etapas seguintes